# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mass Spectrometry of Extracellular Vesicles from Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"License: {meta.license}")
print(f"Version: {meta.version}")
print(f"Date Published: {meta.datePublished}\n")
print(f"Keywords: {meta.keywords}")

## 2. Data Overview
Review available record sets and their fields, showing the `@id` of each.

The Croissant schema may contain several record sets. Let's list all record sets and analyze their fields.

In [ ]:
# List all available record sets by their @id and their fields
record_sets = list(dataset._record_sets_dict.values())  # access private for all declared record sets

if not record_sets:
    print("No record sets found in the Croissant schema. Check schema for dataset definition.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs['@id']}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field['@id']}, dataType: {field.data_type})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the record set and field `@id`s from the overview.

If only a single record set is present (as is common for tabular datasets), it will be automatically loaded.

In [ ]:
# Find tabular record sets
tabular_record_sets = []
for rs in dataset._record_sets_dict.values():
    # Try to pick the main tabular set (with typical field definitions)
    tabular_record_sets.append(rs['@id'])

print(f"Record set IDs available: {tabular_record_sets}\n")

# Load all record sets into DataFrames
dataframes = {}

for rs_id in tabular_record_sets:
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df
    print(f"Loaded record set '{rs_id}' with shape {df.shape} and columns: {df.columns.tolist()}")

# For demonstration, pick the first (main) tabular set for exploration
if tabular_record_sets:
    record_set_id = tabular_record_sets[0]
    print(f"\nDisplaying head of DataFrame for {record_set_id}:")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric columns, and grouping data.

For this example, we will select a numeric column (e.g., 'MW_kDa', molecular weight in kDa) if available.

In [ ]:
# Inspect numeric fields
df = dataframes[record_set_id]
print("All columns in record set:", df.columns.tolist())

# Try to find a numeric field by schema name / common heuristic - replace this with the actual @id if known
import numpy as np

# Heuristic: find numeric columns
numeric_field = None
for col in df.select_dtypes(include=[np.number]).columns:
    numeric_field = col
    print(f"First numeric field chosen for demonstration: {numeric_field}")
    break

if numeric_field is None:
    # Try to parse what should be numeric (for text columns with numbers)
    for col in df.columns:
        if df[col].astype(str).str.replace('.', '', 1).str.isdigit().all():
            try:  # Try to cast and see if mostly works
                df[col] = pd.to_numeric(df[col], errors='coerce')
                numeric_field = col
                print(f"Converted column to numeric: {col}")
                break
            except:
                pass

# Pick a group field heuristically, e.g., 'sample' or 'accession' or 'description'
candidates = [c for c in df.columns if ('sample' in c.lower() or 'group' in c.lower() or 'description' in c.lower())]
group_field = candidates[0] if candidates else None
if group_field:
    print(f"Grouping field chosen: {group_field}")
else:
    print("No clear group/categorical field detected. Will skip grouping.")

if numeric_field:
    # Set a threshold at, e.g., 10th percentile
    threshold = df[numeric_field].quantile(0.1)
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (10th percentile):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()])

    # Group by group_field if present
    if group_field and group_field in df:
        # Grouping and aggregating
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Visualize distribution and relationships.

For this dataset, we may visualize the distribution of protein molecular weights and their mean by group if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Distribution plot
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping field is present, show bar plot
    if group_field and group_field in df:
        grouped = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        grouped.plot(kind='bar')
        plt.title(f"Top 10 mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, preview, and analyze the FAIR² mass spectrometry dataset using `mlcroissant`.

- We retrieved all record sets and their field `@id`s (where available).
- The main record set was loaded into a DataFrame for exploration.
- Performed basic EDA: filtering, normalizing, and grouping on numeric data.
- Visualized core distributions (e.g., molecular weight) and summary statistics by group.

For detailed bioinformatics workflows, refer to the [original dataset publication](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
